# PySpark для стажеров: Глубокое погружение в агрегации и оконные функции

**Составители:**
* **Профессор Ричард Фейнман:** Преподаватель. Объясняет фундаментальные принципы распределенных вычислений и внутреннюю логику операций.
* **Senior Big Data Analyst:** Практикующий инженер. Фокус на Catalyst Optimizer, плане выполнения (Execution Plan), Shuffle и оптимизации производительности.
* **ML Researcher:** Специалист по данным. Фокус на инженерии признаков (Feature Engineering), временных утечках (Data Leakage) и подготовке данных для алгоритмов.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Инициализация сессии с базовыми настройками оптимизации
spark = SparkSession.builder \
    .appName("InternAdvancedPySpark") \
    .config("spark.sql.shuffle.partitions", "10") \
    .getOrCreate()

# Реалистичный датасет: транзакции пользователей
data = [
    ("user_1", "2023-10-01 10:00:00", "electronics", 1500.0),
    ("user_1", "2023-10-02 11:30:00", "groceries", 50.0),
    ("user_1", "2023-10-02 11:35:00", "groceries", 20.0),
    ("user_2", "2023-10-01 09:00:00", "clothing", 200.0),
    ("user_2", "2023-10-05 14:00:00", "electronics", 800.0),
    ("user_3", "2023-10-03 16:00:00", "clothing", 120.0),
]
schema = ["user_id", "timestamp", "category", "amount"]
df = spark.createDataFrame(data, schema)
df = df.withColumn("timestamp", F.to_timestamp("timestamp"))
df.show()

## 1. Трансформации уровня строк (Select & Filter)
**Профессор Фейнман:** В контексте распределенных систем `select` (проекция) и `filter` (селекция) — это узкие (narrow) трансформации. Каждая партиция данных обрабатывается независимо, без сетевого обмена с другими узлами. Это линейные операции $O(N)$ по локальным данным.

**Senior Big Data Analyst:** Вы уже знаете, как их писать. Главное для стажера — понимать концепцию **Predicate Pushdown** и **Column Pruning**. Catalyst Optimizer спустит ваши `filter` и `select` на уровень чтения источника (например, Parquet). Читайте только те колонки и строки, которые нужны для последующего `groupBy`, чтобы минимизировать объем данных в памяти перед шаффлом.

In [ ]:
# Узкая трансформация: проекция и селекция
base_df = df.select("user_id", "timestamp", "amount").filter(F.col("amount") > 0)
base_df.show()

## 2. Агрегации и GroupBy: Механика Shuffle
**Профессор Фейнман:** Агрегация требует полного знания о группе. Поскольку данные распределены по кластеру, системе необходимо перераспределить строки так, чтобы все записи с одинаковым ключом (например, `user_id`) оказались на одном физическом узле. Этот процесс топологически эквивалентен операции All-to-All обмена и называется Shuffle. Вы должны осознавать стоимость этой операции в терминах сетевой задержки и дискового I/O.

**Senior Big Data Analyst:** На уровне кода мы используем `groupBy().agg()`. Обратите внимание на оптимизации:
* Часть функций (например, `F.sum`, `F.count`) поддерживают **Map-Side Combine** — Spark локально агрегирует данные на экзекьюторе перед отправкой по сети, радикально снижая объем шаффла.
* Такие функции, как `F.collect_list`, *не* поддерживают Map-Side Combine. Они пересылают все сырые строки. Если у одного пользователя аномально много транзакций (Data Skew), это вызовет `OutOfMemoryError` на одном из экзекьюторов.
* Вместо `countDistinct`, который требует перемещения всех уникальных ключей, используйте `approx_count_distinct`, работающий на алгоритме HyperLogLog. При ошибке в 2-5% вы получаете прирост скорости на порядки.

**ML Researcher:** На этом этапе мы формируем профиль пользователя. Нам интересна дисперсия его трат (`stddev`), общая активность (`count`) и модальность (`collect_list` или агрегация по категориям). Эти агрегаты не имеют временного контекста (они статичны) и подходят для профилирования, но не для предсказания временных рядов.

In [ ]:
user_profile_df = df.groupBy("user_id").agg(
    F.sum("amount").alias("lifetime_value"),
    F.avg("amount").alias("mean_ticket"),
    F.stddev("amount").alias("ticket_std"), # Важная метрика разброса для ML
    F.count("*").alias("transaction_count"),
    F.approx_count_distinct("category").alias("approx_unique_categories"),
    F.collect_list("category").alias("category_history") # Потенциальный вектор для Embedding'а
)
user_profile_df.show(truncate=False)

## 3. Оконные функции (Window Functions): Временные ряды и смещения

**Профессор Фейнман:** В отличие от `groupBy`, оконная функция сохраняет размерность исходного пространства $N$. Мы определяем локальную окрестность (окно) для каждой строки и вычисляем функцию на этом подмножестве. Ключевые параметры окна:
1. `partitionBy` — задает подпространство изоляции (ортогонально другим группам).
2. `orderBy` — вводит отношение порядка внутри группы (строго необходимо для темпоральной логики).
3. **Frame Specification** (`rowsBetween` / `rangeBetween`) — задает границы скользящей окрестности.

**Senior Big Data Analyst:** Окна требуют шаффла по ключу `partitionBy` и локальной сортировки по `orderBy`. 
**Критический момент для стажера:**
Если вы используете `orderBy` без явного указания фрейма (rowsBetween/rangeBetween), Spark по умолчанию применяет фрейм `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`. Это означает вычисление *нарастающим итогом* (от начала группы до текущей строки). Если вы забыли это и просто хотели получить, например, общую сумму по отделу, отсортированную по дате, вы получите баг.

**ML Researcher:** Оконные функции — это базис для формирования темпоральных признаков без утечки данных (Data Leakage). Когда мы прогнозируем событие в момент $t$, мы имеем право использовать только данные из окна $[t - \Delta, t)$. 
С помощью функций `lag` (сдвиг назад) и скользящих средних мы переводим временную зависимость в табличный формат, пригодный для деревьев решений.

In [ ]:
# 1. Базовое окно с сортировкой по времени
window_user_time = Window.partitionBy("user_id").orderBy("timestamp")

# 2. Окно со строгими границами: скользящее окно за предыдущую и текущую строку (сглаживание)
window_rolling_2 = window_user_time.rowsBetween(-1, Window.currentRow)

# 3. Нарастающий итог (используется дефолтный фрейм RANGE UNBOUNDED PRECEDING to CURRENT ROW)
temporal_df = df \
    .withColumn("prev_transaction_amount", F.lag("amount", 1).over(window_user_time)) \
    .withColumn("time_since_last_txn", F.col("timestamp").cast("long") - F.lag("timestamp", 1).over(window_user_time).cast("long")) \
    .withColumn("cumulative_spend", F.sum("amount").over(window_user_time)) \
    .withColumn("rolling_2_avg", F.avg("amount").over(window_rolling_2))

# Заполняем null значения для первых транзакций (важно для ML моделей)
temporal_df = temporal_df.fillna(0, subset=["time_since_last_txn", "prev_transaction_amount"])

temporal_df.orderBy("user_id", "timestamp").show(truncate=False)

## Заключение для стажера
**Senior Big Data Analyst:** Ваш код может работать на сэмпле локально, но упасть на кластере с петабайтами данных. Всегда анализируйте план выполнения (`df.explain()`). Проверяйте, происходит ли шаффл там, где вы его не ожидали. 
**Профессор Фейнман:** Инструмент (PySpark) — это лишь абстракция над фундаментальной теорией графов и распределенных систем. Понимая, как двигаются данные на физическом уровне (Shuffle, Map, Reduce), вы сможете писать оптимальные запросы интуитивно.
**ML Researcher:** И помните про Data Leakage! Оконная функция `lead` смотрит в будущее — никогда не используйте её для генерации признаков обучающей выборки, иначе ваша модель переобучится на "заглядывании в ответ".